In [1]:
import os
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests

NIM_MAHASISWA = "2501010377"
NAMA_MAHASISWA = "Ni Putu Vira Artika Dewi"
EMAIL_MAHASISWA = f"{NIM_MAHASISWA}@instiki.ac.id"

URL_OVERPASS = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": f"INSTIKI-BIFP243-Praktikum/1.0 ({EMAIL_MAHASISWA})"}

QUERY_OVERPASS = """[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;"""

print(" Sel 1 Berhasil: Library diimpor dan parameter utama telah disiapkan.")

 Sel 1 Berhasil: Library diimpor dan parameter utama telah disiapkan.


In [2]:
BASE_DIR = Path(f"akomodasi_kintamani_{NIM_MAHASISWA}")

# Daftar folder wajib sesuai aturan praktikum
FOLDER_LIST = [
    BASE_DIR / "data" / "raw",
    BASE_DIR / "data" / "interim",
    BASE_DIR / "notebooks",
    BASE_DIR / "docs",
    BASE_DIR / "reports"
]

# Buat direktori jika belum ada
for folder in FOLDER_LIST:
    folder.mkdir(parents=True, exist_ok=True)

print(f" Sel 2 Berhasil: Folder utama '{BASE_DIR}' beserta seluruh sub-foldernya telah dibuat!")


 Sel 2 Berhasil: Folder utama 'akomodasi_kintamani_2501010377' beserta seluruh sub-foldernya telah dibuat!


In [3]:
def ambil_overpass(query, maks_percobaan=4, jeda=5):
    """
    Mengambil data dari Overpass API dengan fitur retry otomatis dan exponential backoff.
    """
    for percobaan in range(1, maks_percobaan + 1):
        try:
            print(f"📡 Mengirim request ke Overpass API (Percobaan {percobaan}/{maks_percobaan})...")
            r = requests.post(URL_OVERPASS, data={"data": query}, headers=HEADERS, timeout=120)

            if r.status_code == 200:
                print(" Pengambilan data sukses (HTTP 200 OK)!")
                return r.json()

            print(f" [Percobaan {percobaan}] Status HTTP {r.status_code}, menunggu {jeda} detik...")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[Percobaan {percobaan}] Error Jaringan: {type(e).__name__}, menunggu {jeda} detik...")

        # Exponential backoff (jeda waktu melipat ganda: 5s, 10s, 20s...)
        time.sleep(jeda)
        jeda *= 2

    raise RuntimeError(" Gagal mengambil data dari Overpass API setelah batas maksimum percobaan.")

print(" Sel 3 Berhasil: Fungsi `ambil_overpass` siap digunakan.")


 Sel 3 Berhasil: Fungsi `ambil_overpass` siap digunakan.


In [4]:
# Jalankan fungsi akuisisi
raw_data = ambil_overpass(QUERY_OVERPASS)

# Buat timestamp unik untuk nama berkas raw
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_filename = f"osm_akomodasi_kintamani_{timestamp}.json"
raw_path = BASE_DIR / "data" / "raw" / raw_filename

# Simpan data JSON mentah utuh
raw_path.write_text(json.dumps(raw_data, ensure_ascii=False, indent=2), encoding="utf-8")

total_elemen = len(raw_data.get("elements", []))
print(f"\n Raw JSON berhasil disimpan di: {raw_path}")
print(f" Total elemen akomodasi yang didapat: {total_elemen} objek.")

📡 Mengirim request ke Overpass API (Percobaan 1/4)...
 [Percobaan 1] Status HTTP 504, menunggu 5 detik...
📡 Mengirim request ke Overpass API (Percobaan 2/4)...
 Pengambilan data sukses (HTTP 200 OK)!

 Raw JSON berhasil disimpan di: akomodasi_kintamani_2501010377/data/raw/osm_akomodasi_kintamani_20261111_044656.json
 Total elemen akomodasi yang didapat: 32 objek.


In [5]:
def ratakan(elements):
    """
    Meratakan struktur hirarki JSON Overpass menjadi DataFrame tabular.
    Atribut nomor telepon dan email sengaja TIDAK diambil (Prinsip Minimisasi Data).
    """
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)  # Jika poligon/way pakai titik tengah (center), jika titik pakai lat/lon langsung

        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}"
        })
    return pd.DataFrame(baris)

print(" Sel 5 Berhasil: Fungsi `ratakan` siap digunakan.")


 Sel 5 Berhasil: Fungsi `ratakan` siap digunakan.


In [6]:
# 1. Ubah elemen JSON menjadi DataFrame
df_akomodasi = ratakan(raw_data["elements"])

print("================ HASIL PEMERIKSAAN KUALITAS DATA ================")

# 2. Periksa dan hilangkan duplikat berdasarkan osm_type + osm_id
jumlah_duplikat = df_akomodasi.duplicated(subset=["osm_type", "osm_id"]).sum()
print(f"1. Objek Duplikat (osm_type + osm_id) : {jumlah_duplikat}")
if jumlah_duplikat > 0:
    df_akomodasi = df_akomodasi.drop_duplicates(subset=["osm_type", "osm_id"])

# 3. Hitung akomodasi tanpa nama
tanpa_nama = df_akomodasi["nama"].isna().sum()
print(f"2. Objek Tanpa Nama (Missing Name)   : {tanpa_nama} (Tetap dipertahankan)")

# 4. Filter pengecekan koordinat out-of-bounds Kintamani (BBox: Lat -8.35 s/d -8.12, Lon 115.28 s/d 115.48)
out_bounds = df_akomodasi[
    ~df_akomodasi["lat"].between(-8.35, -8.12) | ~df_akomodasi["lon"].between(115.28, 115.48)
]
print(f"3. Koordinat di luar Kintamani      : {len(out_bounds)} objek")

# 5. Tampilkan ringkasan jenis akomodasi
print("\n--- Ringkasan Jumlah per Jenis Akomodasi ---")
print(df_akomodasi["jenis"].value_counts())
print("================================================================")

================ HASIL PEMERIKSAAN KUALITAS DATA ================
1. Objek Duplikat (osm_type + osm_id) : 0
2. Objek Tanpa Nama (Missing Name)   : 1 (Tetap dipertahankan)
3. Koordinat di luar Kintamani      : 1 objek

--- Ringkasan Jumlah per Jenis Akomodasi ---
jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64


In [7]:
interim_path = BASE_DIR / "data" / "interim" / "akomodasi_kintamani.csv"

# Simpan dataset ke CSV dengan encoding UTF-8
df_akomodasi.to_csv(interim_path, index=False, encoding="utf-8")

print(f" Dataset interim berhasil disimpan di: {interim_path}")
print(f" Total baris dalam CSV interim: {len(df_akomodasi)} baris\n")

# Tampilkan 5 sampel data pertama di notebook
print("Preview 5 Data Teratas:")
df_akomodasi.head()


 Dataset interim berhasil disimpan di: akomodasi_kintamani_2501010377/data/interim/akomodasi_kintamani.csv
 Total baris dalam CSV interim: 32 baris

Preview 5 Data Teratas:


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
0,node,1977779907,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,None,None,None,http://lakeviewbali.com/en/,https://www.openstreetmap.org/node/1977779907
1,node,2107226701,Arlina,guest_house,-8.249815,115.398465,None,None,None,None,https://www.openstreetmap.org/node/2107226701
2,node,2107227001,Toya Devasya Resort,hotel,-8.250164,115.399798,None,None,None,None,https://www.openstreetmap.org/node/2107227001
3,node,4150326329,Guesthouse,hotel,-8.281456,115.362223,None,None,None,None,https://www.openstreetmap.org/node/4150326329
4,node,4297310696,The Cave Hotel,hotel,-8.280748,115.362048,None,2,5,None,https://www.openstreetmap.org/node/4297310696


In [8]:
import hashlib
import json
from datetime import datetime, timezone

print("🔄 Menyusun seluruh berkas dokumentasi proyek...")

# 1. Buat file docs/perbandingan_bps.csv (Hari 5)
total_osm = len(df_akomodasi)
total_bps = 205  # Angka resmi pembanding dari BPS Bangli / Kintamani Dalam Angka 2024
persentase = round((total_osm / total_bps) * 100, 2)

bps_csv_content = f"""kategori,jumlah_osm,jumlah_bps,persentase_kelengkapan,tahun_data_bps,keterangan
TOTAL AKOMODASI,{total_osm},{total_bps},{persentase}%,2024,Data OSM mencakup {persentase}% dari total data resmi BPS Kintamani
"""
(BASE_DIR / "docs" / "perbandingan_bps.csv").write_text(bps_csv_content, encoding="utf-8")
print("  - docs/perbandingan_bps.csv: OK")

# 2. Buat file docs/query_overpass.txt
(BASE_DIR / "docs" / "query_overpass.txt").write_text(QUERY_OVERPASS, encoding="utf-8")
print("  - docs/query_overpass.txt: OK")

# 3. Buat file docs/provenance.json
with open(raw_path, "rb") as f:
    md5_hash = hashlib.md5(f.read()).hexdigest()

provenance_data = {
    "endpoint_url": URL_OVERPASS,
    "query": QUERY_OVERPASS,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
     "total_elements": len(raw_data.get("elements", [])),
    "raw_filename": raw_path.name,
    "raw_md5_hash": md5_hash,
    "license": "Open Data Commons Open Database License (ODbL)",
    "collector_name": NAMA_MAHASISWA,
    "collector_nim": NIM_MAHASISWA
}
with open(BASE_DIR / "docs" / "provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance_data, f, indent=2)
print("  - docs/provenance.json: OK")

# 4. Buat file docs/data_dictionary.csv
data_dict = """kolom,tipe,deskripsi,contoh,boleh_kosong
osm_type,teks,Jenis objek OSM (node/way/relation),way,Tidak
osm_id,bilangan bulat,ID unik objek OpenStreetMap,123456789,Tidak
nama,teks,Nama tempat akomodasi dari tag name,Batur View Guest House,Ya
jenis,teks,Kategori akomodasi dari tag tourism,guest_house,Tidak
lat,desimal,Koordinat lintang WGS84 titik/center,-8.27123,Tidak
lon,desimal,Koordinat bujur WGS84 titik/center,115.37123,Tidak
desa,teks,Nama desa dari tag addr:village,Central Batur,Ya
bintang,teks,Klasifikasi bintang dari tag stars,3,Ya
jumlah_kamar,teks,Jumlah unit/kamar dari tag rooms,10,Ya
website,teks,URL situs resmi akomodasi,https://baturview.com,Ya
url_osm,teks,Tautan langsung ke objek openstreetmap.org,https://www.openstreetmap.org/way/123456789,Tidak
"""
(BASE_DIR / "docs" / "data_dictionary.csv").write_text(data_dict, encoding="utf-8")
print("  - docs/data_dictionary.csv: OK")

# 5. Buat file docs/data_ethics_checklist.csv
ethics_checklist = """prinsip_etika,status_kepatuhan,bukti_pelaksanaan
API Limit & Respect,Penuh,Menggunakan User-Agent khusus dan delay retry backoff (maks 4x)
Minimisasi Data,Penuh,Nomor telepon dan email pemilik dibuang/tidak diambil dalam script
Lisensi & Atribusi,Penuh,Mencantumkan atribusi © OpenStreetMap contributors (ODbL)
Sumber Terlarang,Penuh,Tidak melakukan scraping ke Traveloka Booking Agoda Airbnb atau Google Maps
Kebenaran Data,Penuh,Raw data JSON tidak diubah dan disimpan secara eksplisit di data/raw/
"""
(BASE_DIR / "docs" / "data_ethics_checklist.csv").write_text(ethics_checklist, encoding="utf-8")
print("  - docs/data_ethics_checklist.csv: OK")

# 6. Buat file README.md utama
readme_content = f"""# Akuisisi Data Akomodasi di Kecamatan Kintamani

**Mata Kuliah:** BIFP-243 Akuisisi dan Manajemen Data
**Pengambil Data:** {NAMA_MAHASISWA} (NIM: {NIM_MAHASISWA})
**Tanggal Akses:** {datetime.now().strftime("%Y-%m-%d")}

## 1. Deskripsi Proyek
Proyek ini bertujuan untuk mengumpulkan, meratakan, dan mendokumentasikan data akomodasi di Kecamatan Kintamani, Kabupaten Bangli, Bali secara etis.

## 2. Sumber Data & Atribusi
* **OpenStreetMap API (Overpass API)** - Data spasial dan atribut akomodasi.
* Data ini berlisensi Open Data Commons Open Database License (ODbL).
* **Atribusi Mandatory:** Data memuat informasi dari OpenStreetMap yang tersedia di bawah lisensi ODbL. **© OpenStreetMap contributors**.

## 3. Batasan Etika
* Mematuhi rate limit Overpass API tanpa request berulang berlebihan.
* Melakukan **minimisasi data** dengan sengaja tidak mengumpulkan nomor telepon, email, dan akun media sosial pemilik.
* Bebas dari scraping platform OTA terlarang (Airbnb, Booking.com, Agoda, dll.).

## 4. Cara Menjalankan Ulang Kode
1. Buka Notebook di Google Colab.
2. Jalankan seluruh sel dari atas ke bawah.
3. Respons mentah tersimpan di `data/raw/` dan dataset interim bersih di `data/interim/akomodasi_kintamani.csv`.

## 5. Struktur Repositori
- `data/` : Menyimpan data raw dan interim
- `docs/` : Menyimpan dokumen analisis dan metadata
"""
(BASE_DIR / "README.md").write_text(readme_content, encoding="utf-8")
print(" Berhasil: Semua berkas dokumentasi proyek telah diperbarui!")


🔄 Menyusun seluruh berkas dokumentasi proyek...
  - docs/perbandingan_bps.csv: OK
  - docs/query_overpass.txt: OK
  - docs/provenance.json: OK
  - docs/data_dictionary.csv: OK
  - docs/data_ethics_checklist.csv: OK
 Berhasil: Semua berkas dokumentasi proyek telah diperbarui!


In [9]:
# SEL 9 (VERSI TABEL): 10 SAMPEL UNTUK VERIFIKASI MANUAL (HARI 5)

# 1. Ambil 10 sampel acak yang memiliki nama
sampel_10 = df_akomodasi.dropna(subset=["nama"]).sample(n=10, random_state=42).copy()

# 2. Pilih dan rapikan kolom yang ingin ditampilkan dalam tabel
tabel_sampel = sampel_10[["osm_id", "nama", "jenis", "desa", "lat", "lon", "url_osm"]].reset_index(drop=True)

# 3. Tampilkan sebagai DataFrame Pandas (Tabel Rapi)
print("====================== 10 SAMPEL DATA VERIFIKASI MANUAL ======================")
display(tabel_sampel)


====================== 10 SAMPEL DATA VERIFIKASI MANUAL ======================


,osm_id,nama,jenis,desa,lat,lon,url_osm
0,11010451459,Manik Tirta Cabin's,hotel,None,-8.249347,115.396872,https://www.openstreetmap.org/node/11010451459
1,6451134585,Caldera,hotel,None,-8.279139,115.367364,https://www.openstreetmap.org/node/6451134585
2,9555392317,Pondok Bali Cottage,guest_house,None,-8.222104,115.398161,https://www.openstreetmap.org/node/9555392317
3,7058054537,Segara Hotel,hotel,None,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537
4,5065920543,The Volcania Guesthouse,hotel,None,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543
5,5175562421,Entrance to: Bunbulan Hill Hostel,hostel,None,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421
6,13068182153,Batur Sunrise Guesthouse,hotel,None,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153
7,9769685089,Bunbulan Panorama Villas,guest_house,None,-8.222736,115.397971,https://www.openstreetmap.org/node/9769685089
8,5479746121,Hut,chalet,None,-8.242693,115.387587,https://www.openstreetmap.org/node/5479746121
9,1977779907,The Lakeview Hotel Bali,hotel,None,-8.283140,115.365844,https://www.openstreetmap.org/node/1977779907


In [14]:
import os
from pathlib import Path


NIM_MAHASISWA = "2501010377"

folder_target = Path(f"akomodasi_kintamani_{NIM_MAHASISWA}")
zip_name = f"akomodasi_kintamani_{NIM_MAHASISWA}.zip"
laporan_pdf_path = folder_target / "reports" / "laporan_akomodasi_kintamani.pdf"

print("🔍 Memeriksa kelengkapan file sebelum di-zip...")

# 1. Cek apakah folder utama ada
if not folder_target.exists():
    print(f" ERROR: Folder '{folder_target}' tidak ditemukan! Pastikan Anda sudah menjalankan Sel 2 (Setup Direktori).")
else:
    # 2. Cek apakah file Laporan PDF sudah diunggah/dimasukkan ke folder reports
    if not laporan_pdf_path.exists():
        print(f" PERINGATAN: File '{laporan_pdf_path}' belum ditemukan di folder reports/!")
        print(" Silakan upload/masukkan file PDF laporan Anda ke folder 'reports/' terlebih dahulu agar tidak ada pengurangan nilai.")
        print("Proses zipping tetap dilanjutkan...\n")
    else:
        print(" File laporan PDF ditemukan di folder reports/!\n")

    # 3. Jalankan kompresi folder ke format ZIP
    print(f" Mengompres folder '{folder_target}' menjadi '{zip_name}'...")
    !zip -r {zip_name} {folder_target}
    print("\n" + "=" * 60)
    print(f" BERHASIL! File '{zip_name}' telah selesai dibuat ulang.")
    print(" Anda dapat mengunduh file ZIP ini melalui panel file (sebelah kiri Colab) atau menyalin tautan repositori untuk dikumpulkan ke elsa.instiki.ac.id.")
    print("=" * 60)


🔍 Memeriksa kelengkapan file sebelum di-zip...
 PERINGATAN: File 'akomodasi_kintamani_2501010377/reports/laporan_akomodasi_kintamani.pdf' belum ditemukan di folder reports/!
 Silakan upload/masukkan file PDF laporan Anda ke folder 'reports/' terlebih dahulu agar tidak ada pengurangan nilai.
Proses zipping tetap dilanjutkan...

 Mengompres folder 'akomodasi_kintamani_2501010377' menjadi 'akomodasi_kintamani_2501010377.zip'...
  adding: akomodasi_kintamani_2501010377/ (stored 0%)
  adding: akomodasi_kintamani_2501010377/docs/ (stored 0%)
  adding: akomodasi_kintamani_2501010377/docs/data_dictionary.csv (deflated 45%)
  adding: akomodasi_kintamani_2501010377/docs/perbandingan_bps.csv (deflated 18%)
  adding: akomodasi_kintamani_2501010377/docs/query_overpass.txt (deflated 21%)
  adding: akomodasi_kintamani_2501010377/docs/provenance.json (deflated 33%)
  adding: akomodasi_kintamani_2501010377/docs/data_ethics_checklist.csv (deflated 33%)
  adding: akomodasi_kintamani_2501010377/reports/ (